# rai-microrts-arena Kaggle T4x2 训练脚本

这个 notebook 只使用 Kaggle 自带环境：直接使用当前 notebook 的 Python、CUDA 和 Torch。项目元数据里的 `python <3.12` 限制不会再触发虚拟环境分支；Python >= 3.12 时直接通过 `PYTHONPATH` 使用源码。

执行流程：

1. 检查 GPU、Java、Python 和 CUDA。
2. 使用 Kaggle 当前 kernel，不创建额外 Python 环境，不安装 uv。
3. 安装 SB3、microRTS 相关依赖；没有可用 wheel 的包会尝试源码编译。
4. 克隆/更新仓库并追加 Kaggle 友好的 PPO 配置。
5. 先 profile `squeeze_unet` 和 `hybrid_entity_grid`，再启动训练。
6. 单卡跑通 baseline 后，再按需启用双 T4 并行实验。

> 这里不是 PPO 内部 DDP。T4x2 更稳的用法是两个独立实验并行：GPU0 跑 SquNet baseline，GPU1 跑 HybridEntityGrid 消融。


## 重点观察指标

**评估/胜负率**：看 `eval/score`、`WinLoss`，并分别关注 Mayari、Coac、WorkerRush、LightRush。只看 shaped reward 容易误判策略质量。

**PPO 稳定性**：看 `approx_kl`、`clip_fraction`、`entropy`、`explained_variance`、`policy_loss`、`value_loss`、`grad_norm`。KL 长期接近 0 通常说明更新太弱；entropy 掉得太快说明探索不足。

**动作 mask/环境健康**：看 `action_mask_stats/valid_locs`、`action_mask_stats/no_valid`、episode length、timeout/truncation。`no_valid` 应该接近 0，valid locs 异常说明 mask 或观测可能坏了。

**吞吐与瓶颈**：看 `steps_per_second`、GPU util、GPU memory，以及 `scripts/profile_microrts_policy.py` 的 `p95_ms`。microRTS 在 Kaggle 上常常是 Java env/CPU 限速，不一定是 GPU 限速。

**调参顺序**：优先调 `n_envs`、`n_steps`、`batch_size`、`n_epochs`、`learning_rate`、`clip_range`、`ent_coef`、`vf_coef`、`gamma`、`gae_lambda`。先稳定 KL 和胜率，再逐步提高吞吐。


In [ ]:
# 0. Runtime sanity check. This is Kaggle's system Python.
import os, sys, subprocess, textwrap, json
from pathlib import Path

print('system python:', sys.version)
print('Kaggle working dir:', Path('/kaggle/working').exists())
print('Kaggle input dir:', Path('/kaggle/input').exists())
!nvidia-smi
!java -version || true


In [ ]:
# 1. User config
from pathlib import Path
import os, sys

REPO_URL = 'https://github.com/SShion0721/rai-microrts-arena.git'
BRANCH = 'main'
WORKDIR = Path('/kaggle/working/rai-microrts-arena')
PYTHON_SENTINEL = Path('/kaggle/working/rai_microrts_python.txt')

# Always use Kaggle's active notebook Python/CUDA/Torch environment.
KAGGLE_PY = Path(sys.executable)

# Start small. Increase to 10e6/20e6 only after the smoke run is healthy.
KAGGLE_TIMESTEPS = '2e6'
WANDB_PROJECT = 'rai-microrts-kaggle'
USE_WANDB = False

os.environ['REPO_URL'] = REPO_URL
os.environ['BRANCH'] = BRANCH
os.environ['WORKDIR'] = str(WORKDIR)
os.environ['PYTHONPATH'] = str(WORKDIR) + ':' + os.environ.get('PYTHONPATH', '')
os.environ['KAGGLE_TIMESTEPS'] = KAGGLE_TIMESTEPS
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
os.environ['WANDB_MODE'] = 'online' if USE_WANDB else 'disabled'
print('WORKDIR =', WORKDIR)
print('KAGGLE_PY =', KAGGLE_PY)
print('PYTHONPATH starts with WORKDIR =', os.environ['PYTHONPATH'].startswith(str(WORKDIR)))
print('WANDB_MODE =', os.environ['WANDB_MODE'])


In [ ]:
%%bash
set -Eeuo pipefail
trap 'rc=$?; echo; echo "FAILED at line $LINENO"; echo "Command: $BASH_COMMAND"; echo "Exit code: $rc"; exit $rc' ERR

# Use Kaggle's existing Python/CUDA/Torch environment. Do not install uv or create another Python environment.
# Clear stale variables left by earlier notebook runs.
unset PYTHON_BIN USE_KAGGLE_ENV || true
export REPO_URL="${REPO_URL:-https://github.com/SShion0721/rai-microrts-arena.git}"
export BRANCH="${BRANCH:-main}"
export WORKDIR="${WORKDIR:-/kaggle/working/rai-microrts-arena}"
export KAGGLE_PY="$(command -v python)"
export PYTHONPATH="$WORKDIR:${PYTHONPATH:-}"
export PATH="$HOME/.local/bin:$PATH"
export MAKEFLAGS="-j$(nproc)"
export CMAKE_BUILD_PARALLEL_LEVEL="$(nproc)"
export PYTHON_SENTINEL="/kaggle/working/rai_microrts_python.txt"

echo "$KAGGLE_PY" > "$PYTHON_SENTINEL"
echo "Selected Kaggle Python: $KAGGLE_PY"
"$KAGGLE_PY" -V
"$KAGGLE_PY" -m pip -V
nvidia-smi || true

# Build/runtime system packages. Kaggle often already has many of these.
if command -v apt-get >/dev/null 2>&1; then
  apt-get update -qq || true
  apt-get install -y -qq \
    default-jdk xvfb ffmpeg git unzip \
    build-essential python3-dev cmake ninja-build swig pkg-config || true
fi
command -v java
java -version

if [ ! -d "$WORKDIR/.git" ]; then
  git clone --branch "$BRANCH" --depth 1 "$REPO_URL" "$WORKDIR"
else
  cd "$WORKDIR"
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull --ff-only || true
fi

cd "$WORKDIR"

# Upgrade only the build frontend. Keep Kaggle's torch/torchvision/CUDA stack as-is.
"$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python -U pip setuptools wheel build cython packaging ninja

# Python >=3.12 cannot satisfy this repo's current pyproject Requires-Python (<3.12).
# Use PYTHONPATH fallback instead of failing editable install.
if "$KAGGLE_PY" - <<'PYCHECK'
import sys
raise SystemExit(0 if sys.version_info < (3, 12) else 1)
PYCHECK
then
  "$KAGGLE_PY" -m pip install --no-cache-dir -e . --no-deps || echo "editable install failed; using PYTHONPATH fallback"
else
  echo "Python >=3.12 detected; skipping editable install and using PYTHONPATH fallback."
fi

# Minimal runtime deps. Do not install torch/torchvision here; use Kaggle's preinstalled CUDA build.
"$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python --prefer-binary \
  'numpy<2' \
  'gymnasium==0.29.1' \
  'stable-baselines3==2.1.0' \
  'ray[air]>=2.8.1,<2.40' \
  'moviepy<2' \
  wandb tensorboard accelerate einops GPUtil \
  pyvirtualdisplay PyYAML tqdm psutil pandas matplotlib

# Native / sometimes-problematic deps. Try wheels first; if unavailable, build locally inside Kaggle.
"$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python --prefer-binary \
  'JPype1>=1.3,<2' 'peewee>=3.14.8' 'PettingZoo==1.24.3' || {
    echo 'Binary install failed; trying local source builds for native deps.'
    "$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python --no-binary=JPype1 'JPype1>=1.3,<2'
    "$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python --no-binary=peewee 'peewee>=3.14.8'
    "$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python 'PettingZoo==1.24.3'
  }

PYTHONPATH="$WORKDIR:${PYTHONPATH:-}" "$KAGGLE_PY" - <<'PYCHK'
import sys
import torch
import gymnasium
import stable_baselines3
import yaml
import jpype
import ray
import rl_algo_impls
print('python executable:', sys.executable)
print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
print('gymnasium:', gymnasium.__version__)
print('stable_baselines3:', stable_baselines3.__version__)
PYCHK


In [ ]:
# 2. Import check in the selected Kaggle Python
from pathlib import Path
KAGGLE_PY = Path('/kaggle/working/rai_microrts_python.txt').read_text(encoding='utf-8').strip()
print('Using Kaggle Python:', KAGGLE_PY)
!cd "$WORKDIR" && PYTHONPATH="$WORKDIR:${PYTHONPATH:-}" $KAGGLE_PY - <<'PY'
import sys
import torch
import gymnasium
import stable_baselines3
import yaml
import jpype
import ray
import rl_algo_impls
print('python executable:', sys.executable)
print('python version:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
print('gymnasium:', gymnasium.__version__)
print('stable_baselines3:', stable_baselines3.__version__)
PY


In [ ]:
# 3. Append Kaggle-friendly PPO configs. This only changes the notebook runtime copy.
from pathlib import Path

hp_path = WORKDIR / 'rl_algo_impls' / 'hyperparams' / 'ppo-Microrts.yml'
text = hp_path.read_text(encoding='utf-8')

KAGGLE_CONFIG = f"""
# ---- Kaggle T4x2 smoke/ablation configs appended by kaggle_microrts_t4x2.ipynb ----
Microrts-kaggle-squnet-map16-bots: &microrts-kaggle-squnet-map16-bots
  <<: *microrts-squnet-map16
  n_timesteps: !!float {KAGGLE_TIMESTEPS}
  evaluate_after_training: true
  # Keep Kaggle smoke runs static; inherited transitions may omit multi_reward_weights.
  hyperparam_transitions_kwargs: {{}}
  device_hyperparams:
    set_float32_matmul_precision: high
    use_deterministic_algorithms: false
    torch_compile: true
    compile_mode: reduce-overhead
  env_hyperparams:
    <<: *microrts-squnet-map16-env-defaults
    n_envs: 12
    self_play_kwargs: null
    map_paths:
      - maps/16x16/basesWorkers16x16A.xml
      - maps/16x16/TwoBasesBarracks16x16.xml
      - maps/16x16/melee16x16Mixed12.xml
    make_kwargs:
      <<: *microrts-squnet-map16-env-make-kwargs-defaults
      num_selfplay_envs: 0
      num_bot_envs: 12
      max_steps: 3000
    bots:
      coacAI: 6
      mayari: 6
  rollout_hyperparams:
    <<: *microrts-ai-rollout-defaults
    n_steps: 512
  algo_hyperparams:
    <<: *microrts-squnet-map16-algo-defaults
    batch_size: 3072
    n_epochs: 4
    learning_rate: !!float 1e-4
    clip_range: 0.1
    ent_coef: 0.01
    # microRTS returns shaped, win/loss, and score-delta rewards. PPO policy loss needs a single advantage.
    multi_reward_weights: [0.8, 0.01, 0.19]
    vf_coef: [0.5, 0.1, 0.2]
  eval_hyperparams:
    <<: *microrts-squnet-map16-eval-defaults
    step_freq: !!float 2.5e5
    n_episodes: 12
    disable_video_generation: true
    env_overrides:
      <<: *microrts-squnet-map16-eval-env-overrides
      n_envs: 12
      self_play_kwargs: {{}}
      bots:
        coacAI: 3
        mayari: 3
        workerRushAI: 3
        lightRushAI: 3

Microrts-kaggle-hybrid-map16-bots:
  <<: *microrts-kaggle-squnet-map16-bots
  policy_hyperparams:
    <<: *microrts-squnet-map16-policy-defaults
    actor_head_style: hybrid_entity_grid
    normalization: layer
    encoder_embed_dim: 128
    encoder_attention_heads: 4
    encoder_feed_forward_dim: 256
    encoder_layers: 2
    actor_head_kernel_size: 3
""".strip() + "\n"

marker = '# ---- Kaggle T4x2 smoke/ablation configs appended by kaggle_microrts_t4x2.ipynb ----'
if marker in text:
    text = text[:text.index(marker)].rstrip()
    print('Replacing existing Kaggle configs in', hp_path)
else:
    text = text.rstrip()
    print('Appending Kaggle configs to', hp_path)
hp_path.write_text(text + "\n\n" + KAGGLE_CONFIG, encoding='utf-8')

print('Configured timesteps:', KAGGLE_TIMESTEPS)


In [ ]:
# 4. Fast architecture profiling before spending hours training
from pathlib import Path
KAGGLE_PY = Path('/kaggle/working/rai_microrts_python.txt').read_text(encoding='utf-8').strip()
!cd "$WORKDIR" && PYTHONPATH="$WORKDIR:${PYTHONPATH:-}" $KAGGLE_PY scripts/profile_microrts_policy.py   --styles squeeze_unet,hybrid_entity_grid   --map-size 16 --batch-size 4 --entities 24   --warmup 5 --iters 20   --action-mode sample


In [ ]:
# 5. Single-GPU baseline training. Start here.
# Watch TensorBoard/W&B while this runs. If it crashes, reduce n_envs in the appended config from 12 to 6.
from pathlib import Path
KAGGLE_PY = Path('/kaggle/working/rai_microrts_python.txt').read_text(encoding='utf-8').strip()
!cd "$WORKDIR" && PYTHONPATH="$WORKDIR:${PYTHONPATH:-}" CUDA_VISIBLE_DEVICES=0 $KAGGLE_PY train.py   --algo ppo   --env Microrts-kaggle-squnet-map16-bots   --seed 1   --device-indexes 0   --wandb-project-name $WANDB_PROJECT   --wandb-tags kaggle t4x2 squnet map16 bots


## 双 T4 并行实验

microRTS 通常会被 CPU/Java 环境拖慢，所以双卡不一定能让单个 PPO run 线性加速。更稳妥的方式是每张 T4 跑一个独立实验。

默认安排：

- GPU0: `Microrts-kaggle-squnet-map16-bots`, seed 1
- GPU1: `Microrts-kaggle-hybrid-map16-bots`, seed 2

如果 Kaggle CPU 撑不住，把 `n_envs` 和 `num_bot_envs` 从 12 改成 6，并把 bots 改成 `coacAI: 3`、`mayari: 3`。


In [ ]:
# 6. Optional dual-GPU parallel run: one experiment per T4.
# Set RUN_DUAL = True when you are ready.
RUN_DUAL = False

from pathlib import Path
KAGGLE_PY = Path('/kaggle/working/rai_microrts_python.txt').read_text(encoding='utf-8').strip()

if RUN_DUAL:
    import subprocess, os, sys
    log_dir = WORKDIR / 'kaggle_logs'
    log_dir.mkdir(exist_ok=True)
    jobs = [
        ('0', 'Microrts-kaggle-squnet-map16-bots', '1', 'squnet'),
        ('1', 'Microrts-kaggle-hybrid-map16-bots', '2', 'hybrid'),
    ]
    procs = []
    for gpu, env_name, seed, label in jobs:
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = gpu
        env['WANDB_MODE'] = 'online' if USE_WANDB else 'disabled'
        env['PYTHONPATH'] = str(WORKDIR) + ':' + env.get('PYTHONPATH', '')
        log_path = log_dir / f'{label}-gpu{gpu}-seed{seed}.log'
        cmd = [
            str(KAGGLE_PY), 'train.py',
            '--algo', 'ppo',
            '--env', env_name,
            '--seed', seed,
            '--device-indexes', '0',
            '--wandb-project-name', WANDB_PROJECT,
            '--wandb-tags', 'kaggle', 't4x2', label, 'map16', 'bots',
        ]
        print('Starting', label, 'on physical GPU', gpu, 'log:', log_path)
        f = open(log_path, 'w')
        procs.append((label, subprocess.Popen(cmd, cwd=WORKDIR, env=env, stdout=f, stderr=subprocess.STDOUT), f))
    for label, proc, f in procs:
        code = proc.wait()
        f.close()
        print(label, 'exit code:', code)
else:
    print('RUN_DUAL is False; skipped.')


In [ ]:
# 7. TensorBoard
# If the Kaggle notebook extension fails, the runs/ folder is still saved as an output artifact.
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/rai-microrts-arena/runs


In [ ]:
# 8. Inspect and package outputs
import os
os.chdir(WORKDIR)
!find saved_models -maxdepth 3 -type f | head -50 || true
!tar -czf /kaggle/working/rai_microrts_outputs.tgz saved_models runs videos kaggle_logs 2>/dev/null || true
print('Packed outputs to /kaggle/working/rai_microrts_outputs.tgz')


## 常见调试方向

- `approx_kl` 过高：把 `learning_rate` 降到 `5e-5`，或把 `n_epochs` 从 4 降到 2。
- `entropy` 很快接近 0：探索塌缩，调高 `ent_coef`，或先延长 bot warmup。
- `explained_variance` 长期很低：critic 学不动，检查 reward scale、`vf_coef` 和 `normalize_value_targets`。
- `steps_per_second` 很低但 GPU 很闲：瓶颈在 CPU/Java env，优先调 `n_envs`，再考虑更轻的 bot/eval 设置。
- Hybrid 比 SquNet 慢很多：先看 profiler 的 p95，再决定是否缩小 attention/head 参数或尝试 Mamba。
- SquNet 和 Hybrid 都学不动：优先考虑 ACBC warm-start、league/PFSP、GraphDINO 或 Mamba entity branch。
